# Modulo 05 - Programmazione Funzionale

---

Python è un linguaggio **multi-paradigma**: lo stesso problema può essere risolto in stile **imperativo** (cicli e condizionali che modificano variabili passo dopo passo), **orientato agli oggetti** (classi che raggruppano dati e comportamento) o **funzionale** (funzioni piccole che trasformano i dati, senza alterare ciò che già esiste). In questo modulo conoscerai lo stile funzionale: prima impari a creare funzioni anonime con `lambda`, poi combini queste funzioni con tre strumenti fondamentali — `map()`, `filter()` e `reduce()` — per trasformare, selezionare e combinare dati di una collezione senza scrivere un ciclo `for` per ogni fase. Il filo conduttore sarà una calcolatrice di interesse composto e l'elaborazione di liste di e-mail e numeri.

Corso: Ready To Deploy

Creato da: [Enzo Schitini](https://www.linkedin.com/in/enzoschitini)

---

## Argomenti

| **Argomento** | Descrizione |
| --- | --- |
| 1. Funzione lambda | Funzioni anonime di una riga, buone pratiche e funzioni di ordine superiore, applicate a un calcolatore di interessi. |
| 2. Funzione map | Trasforma tutti gli elementi di una collezione, applicata all'estrazione di provider di e-mail e a scenari di investimento. |
| 3. Funzione filter | Seleziona solo gli elementi che soddisfano una condizione, applicata al filtraggio delle e-mail di un provider. |
| 4. Funzione reduce | Combina tutti gli elementi di una collezione in un unico valore, applicata alla ricerca del numero più grande di una lista. |

---

## 1. Funzione lambda

Il primo passo della programmazione funzionale è riuscire a creare funzioni piccole e usa e getta senza tutta la cerimonia di un `def`. È per questo che esiste la funzione **`lambda`**.

### 1.1 Definizione

Una **funzione `lambda`** è una funzione **anonima** (senza nome), scritta in un'unica riga, contenente solo **un'espressione** — non può avere cicli, `print()` o istruzioni multiple. La sua sintassi è:

```python
variabile = lambda parametri: espressione
```

L'`espressione` è automaticamente il valore di ritorno, senza bisogno della parola chiave `return`.

**Esempio:** estraendo il provider da un'e-mail.

In [1]:
extrair_provedor_email = lambda email: email.split(sep='@')[-1]

In [2]:
email = 'andre.perez@gmail.com'
print(email)

provedor_email = extrair_provedor_email(email)
print(provedor_email)

andre.perez@gmail.com
gmail.com


### 1.2 Buone pratiche

Poiché una `lambda` può contenere solo un'espressione, è comune usare l'**operatore ternario** (`valore_se_vero if condizione else valore_se_falso`) per inserire una decisione semplice.

**Esempio:** verificando se un numero è pari.

In [3]:
# Funziona, ma l'if/else è ridondante: il confronto è già un booleano
numero_e_par_verboso = lambda numero: True if numero % 2 == 0 else False

# Forma diretta e più idiomatica: basta restituire il confronto stesso
numero_e_par = lambda numero: numero % 2 == 0

print(numero_e_par_verboso(4))
print(numero_e_par(4))

True
True


In [4]:
for numero in range(10):
    if numero_e_par(numero):
        print(f'Il numero {numero} è pari!')

O número 0 é par!
O número 2 é par!
O número 4 é par!
O número 6 é par!
O número 8 é par!


> ⚠️ **Attenzione:** essendo limitata a un'espressione, `lambda` non sostituisce `def` in ogni situazione. Usa `lambda` per logiche brevi e, soprattutto, passate direttamente come argomento di un'altra funzione (come vedremo in `map`, `filter` e `reduce`). Per una logica con più righe o che necessita di un nome proprio nel codice, preferisci `def` — la PEP 8 raccomanda addirittura di non assegnare una `lambda` a una variabile quando un `def` renderebbe il codice più chiaro.

### 1.3 Funzioni di ordine superiore

Una **funzione di ordine superiore** è una funzione che **riceve un'altra funzione come parametro** o che **restituisce una funzione**. Combinare questo con `lambda` è molto potente: una funzione può "fabbricare" altre funzioni su misura.

**Esempio:** una calcolatrice di rendimento di investimento, in cui il tasso di interesse è già incorporato nella funzione generata.

In [5]:
def criar_calculadora_retorno(taxa_juros: float):
    # La lambda restituita "ricorda" il valore di taxa_juros anche dopo
    # che criar_calculadora_retorno() ha già terminato l'esecuzione — questo
    # si chiama closure.
    return lambda investimento: investimento * (1 + taxa_juros)

In [6]:
retorno_5_porcento = criar_calculadora_retorno(taxa_juros=0.05)
retorno_10_porcento = criar_calculadora_retorno(taxa_juros=0.10)

In [7]:
# Ogni calcolatrice già "conosce" il proprio tasso di interesse
valor_final = retorno_5_porcento(investimento=1000)
print(round(valor_final, 2))

valor_final = retorno_10_porcento(investimento=1000)
print(round(valor_final, 2))

1050.0
1100.0


Poiché le calcolatrici restituiscono un valore, possiamo **incatenarle in un ciclo** per simulare l'interesse composto lungo diversi anni:

In [8]:
anos = 10
valor_final = 1000

for _ in range(anos):
    valor_final = retorno_5_porcento(investimento=valor_final)

print(round(valor_final, 2))

1628.89


In [9]:
anos = 10
valor_final = 1000

for _ in range(anos):
    valor_final = retorno_10_porcento(investimento=valor_final)

print(round(valor_final, 2))

2593.74


---

## 2. Funzione map

Con `lambda` già padroneggiata, possiamo combinarla con funzioni che operano su **collezioni intere** in un colpo solo. La prima è `map()`, che trasforma ogni elemento di una collezione.

### 2.1 Definizione

La funzione `map()` applica una funzione a **tutti** gli elementi di una collezione (`list`, `dict`, ecc.) e restituisce **tutti** gli elementi già trasformati:

```python
variabile = map(funzione, collezione)
```

In [10]:
numeros = [1, 2, 3]

numeros_ao_cubo = map(lambda num: num ** 3, numeros)
print(numeros_ao_cubo)

> ⚠️ **Attenzione:** `map()` non restituisce una `list`, bensì un **iteratore** — per questo `print()` mostra solo l'indirizzo dell'oggetto in memoria, e non i valori. Per vedere o usare i risultati, converti con `list()`. Inoltre, un iteratore può essere **percorso una sola volta**: dopo essere stato convertito (o usato in un ciclo), rimane "vuoto".

In [11]:
print(list(numeros_ao_cubo))

[1, 8, 27]


### 2.2 Sostituendo i cicli con map

**Esempio:** estraendo il provider da più e-mail. Prima, la forma imperativa, con un ciclo `for`:

In [12]:
emails = ['andre.perez@gmail.com', 'andre.perez@live.com', 'andre.perez@yahoo.com']
extrair_provedor_email = lambda email: email.split(sep='@')[-1]

In [13]:
# Forma imperativa: costruiamo la lista manualmente, elemento per elemento
provedores = []
for email in emails:
    provedor = extrair_provedor_email(email)
    provedores.append(provedor)

print(provedores)

['gmail.com', 'live.com', 'yahoo.com']


Ora, lo stesso compito in stile funzionale, con `map()` — senza ciclo, senza lista vuota da riempire:

In [14]:
provedores = list(map(extrair_provedor_email, emails))
print(provedores)

['gmail.com', 'live.com', 'yahoo.com']


> 💡 **Consiglio:** quando la funzione viene usata in un solo posto, non c'è bisogno di nominarla — possiamo passare la `lambda` direttamente come argomento di `map()`.

In [15]:
provedores = list(map(lambda email: email.split(sep='@')[-1], emails))
print(provedores)

['gmail.com', 'live.com', 'yahoo.com']


### 2.3 map con più parametri

`map()` accetta anche **più di una collezione**: in questo caso, la funzione passata deve ricevere un parametro per ogni collezione, e gli elementi vengono combinati posizione per posizione (il 1º elemento di ogni lista, poi il 2º, e così via).

**Esempio:** calcolando il rendimento di più scenari di investimento in un colpo solo.

In [16]:
def calcular_retorno_investimento(valor_inicial: float, taxa_juros: float, anos: int) -> float:
    valor_final = valor_inicial
    for _ in range(anos):
        valor_final = valor_final * (1 + taxa_juros)
    return round(valor_final, 2)

In [17]:
valores_iniciais = [1000, 1000, 1000]
taxas_juros = [0.05, 0.10, 0.15]
anos = [10, 10, 10]

cenarios = list(map(calcular_retorno_investimento, valores_iniciais, taxas_juros, anos))
print(cenarios)

[1628.89, 2593.74, 4045.56]


---

## 3. Funzione filter

Mentre `map()` trasforma tutti gli elementi, a volte vogliamo solo **selezionarne** alcuni. Per questo esiste `filter()`.

### 3.1 Definizione

La funzione `filter()` applica una funzione **logica** (che restituisce `True` o `False`) a tutti gli elementi di una collezione, e restituisce **solo** gli elementi per cui il risultato è stato `True`:

```python
variabile = filter(funzione_logica, collezione)
```

Come `map()`, anche il ritorno di `filter()` è un **iteratore**.

In [18]:
numeros = [1, 2, 3, 4, 5, 6]

numeros_pares = filter(lambda num: num % 2 == 0, numeros)
print(list(numeros_pares))

[2, 4, 6]


### 3.2 Sostituendo i cicli con filter

**Esempio:** selezionando solo le e-mail di un provider specifico. Di nuovo, confrontando la forma imperativa con quella funzionale.

In [19]:
emails = ['andre.perez@gmail.com', 'andre.perez@live.com', 'andre.perez@yahoo.com']
eh_do_gmail = lambda email: 'gmail' in email

In [20]:
# Forma imperativa
emails_gmail = []
for email in emails:
    if eh_do_gmail(email):
        emails_gmail.append(email)

print(emails_gmail)

['andre.perez@gmail.com']


In [21]:
# Forma funzionale, equivalente
emails_gmail = list(filter(eh_do_gmail, emails))
print(emails_gmail)

['andre.perez@gmail.com']


> 💡 **Consiglio:** il criterio del filtro raramente viene riutilizzato in un altro punto del codice — per questo, è molto comune scriverlo direttamente come una `lambda`, senza nominarlo prima.

In [22]:
emails_gmail = list(filter(lambda email: 'gmail' in email, emails))
print(emails_gmail)

['andre.perez@gmail.com']


---

## 4. Funzione reduce

`map()` trasforma e `filter()` seleziona, ma entrambi restituiscono ancora una **collezione**. Quando l'obiettivo è condensare tutto in un **unico valore** — una somma, l'elemento più grande, una concatenazione — usiamo `reduce()`.

### 4.1 Definizione

La funzione `reduce()` applica una funzione a tutti gli elementi di una collezione, **due alla volta**, accumulando il risultato, finché non rimane **un unico valore**:

```python
variabile = reduce(funzione, collezione)
```

> ⚠️ **Attenzione:** a differenza di `map()` e `filter()`, `reduce()` non è una funzione nativa di Python — dalla versione 3 in poi deve essere importata dal modulo `functools`.

In [23]:
from functools import reduce

numeros = [1, 2, 3, 4]

soma = reduce(lambda acumulado, atual: acumulado + atual, numeros)
print(soma)

10


### 4.2 Funzioni di ordine superiore con reduce

**Esempio:** trovando il numero più grande di una lista, senza usare la funzione pronta `max()`.

In [24]:
def maior_entre(primeiro: int, segundo: int) -> int:
    return primeiro if primeiro >= segundo else segundo

print(maior_entre(11, 4))

11


In [25]:
from random import random

# Lista con 100 numeri interi casuali tra 0 e 100
# (list comprehension: vedremo questa sintassi in dettaglio in un modulo futuro)
numeros = [round(100 * random()) for _ in range(100)]
print(numeros[:10], '...')  # mostrando solo i primi 10

[47, 21, 4, 15, 42, 56, 60, 35, 71, 84] ...


In [26]:
# reduce applica maior_entre() ai primi due numeri, poi al
# risultato con il terzo, poi al risultato con il quarto, e così
# via, finché non rimane solo il più grande di tutti
maior_numero = reduce(maior_entre, numeros)
print(maior_numero)

100


In [27]:
# La stessa logica, con la funzione scritta come lambda
maior_numero = reduce(lambda primeiro, segundo: primeiro if primeiro >= segundo else segundo, numeros)
print(maior_numero)

100


> 💡 **Consiglio:** per trovare il valore più grande di una collezione, Python ha già la funzione nativa `max()` — molto più diretta di `reduce()` per questo caso specifico. L'esempio sopra ha fini didattici: `reduce()` si distingue davvero quando la logica di combinazione è **personalizzata** e non esiste una funzione pronta per essa.

### 4.3 Combinando map, filter e reduce

Poiché `map()`, `filter()` e `reduce()` seguono lo stesso schema (funzione + collezione), possono essere **incatenati**: l'output di uno diventa l'input del successivo.

**Esempio:** sommando il quadrato solo dei numeri originariamente dispari.

In [28]:
from random import random

numeros = [round(100 * random()) for _ in range(100)]
print(numeros[:10], '...')

[50, 3, 65, 46, 43, 99, 40, 69, 84, 98] ...


**Passo 1:** elevare ogni numero al quadrato (`map`).

In [29]:
numeros_ao_quadrado = map(lambda numero: numero ** 2, numeros)

**Passo 2:** mantenere solo i quadrati dei numeri dispari — il quadrato di un numero dispari è anch'esso dispari (`filter`).

In [30]:
quadrados_impares = filter(lambda numero: numero % 2 != 0, numeros_ao_quadrado)

**Passo 3:** sommare tutto in un unico valore (`reduce`).

In [31]:
soma_quadrados_impares = reduce(lambda acumulado, atual: acumulado + atual, quadrados_impares)
print(soma_quadrados_impares)

190475


> ⚠️ **Attenzione:** `map` e `filter` restituiscono iteratori che possono essere percorsi **una sola volta**. Per questo, l'esempio sotto ricomincia dalla lista `numeros` originale — riutilizzare `numeros_ao_quadrado` o `quadrados_impares` qui non funzionerebbe, poiché entrambi sono già stati consumati nel passo 3 sopra.

In [32]:
soma_quadrados_impares = reduce(
    lambda acumulado, atual: acumulado + atual,
    filter(
        lambda numero: numero % 2 != 0,
        map(lambda numero: numero ** 2, numeros),
    ),
)
print(soma_quadrados_impares)

190475


> 💡 Anche se l'incatenamento sopra funziona su un'unica riga, diventa difficile da leggere. Nella pratica quotidiana, preferisci il formato passo dopo passo (come abbiamo fatto prima) ogni volta che la riga diventa troppo lunga — il codice funzionale non deve essere compatto per essere buono.

---

## Riepilogo del Modulo

| Strumento | Cosa fa | Restituisce | Esempio |
| --- | --- | --- | --- |
| `lambda` | Crea una funzione anonima di una sola espressione | Funzione | `lambda x: x ** 2` |
| `map(f, collezione)` | Trasforma ogni elemento della collezione | Iteratore | `map(lambda x: x * 2, numeri)` |
| `filter(f, collezione)` | Mantiene solo gli elementi in cui `f` restituisce `True` | Iteratore | `filter(lambda x: x > 0, numeri)` |
| `reduce(f, collezione)` | Combina tutti gli elementi in un unico valore (richiede `from functools import reduce`) | Valore unico | `reduce(lambda a, b: a + b, numeri)` |

Funzioni native e di ordine superiore viste in questo modulo: `lambda`, `map()`, `filter()`, `reduce()` (da `functools`), `list()`, `max()`.